In [ ]:
import serial
import time
import numpy as np
import pandas as pd
from scipy.signal import welch
from sklearn.linear_model import LinearRegression
from collections import deque
import board
import digitalio
import analogio



In [ ]:
S0 = digitalio.DigitalInOut(board.D2)
S1 = digitalio.DigitalInOut(board.D3)
S2 = digitalio.DigitalInOut(board.D4)

S0.direction = digitalio.Direction.OUTPUT
S1.direction = digitalio.Direction.OUTPUT
S2.direction = digitalio.Direction.OUTPUT

S0.value = False
S1.value = False
S2.value = False

In [35]:
# -------------------------------------------------
# PARAMETERS
# -------------------------------------------------
PORT = "COM4"         # change if needed
BAUD = 115200
FS = 200
WINDOW = 200          # 1 second window

CSV_FILE = "data/Combined_BL50.csv"

# -------------------------------------------------
# FEATURE FUNCTIONS
# -------------------------------------------------


In [36]:
def compute_rms(signal):
    return np.sqrt(np.mean(signal ** 2))

def compute_mdf(signal, fs):
    f, Pxx = welch(signal, fs=fs, nperseg=len(signal))
    cumsum = np.cumsum(Pxx)
    return f[np.where(cumsum >= cumsum[-1] / 2)[0][0]]

# -------------------------------------------------
# LOAD DATASET & BUILD REFERENCE
# -------------------------------------------------
df = pd.read_csv(CSV_FILE)

rms_list = []
mdf_list = []

step = WINDOW
n_windows = (len(df) - WINDOW) // step + 1

for w in range(n_windows):
    start = w * step
    raw = df["Raw_EMG"].iloc[start:start+WINDOW].values
    env = df["Envelope_EMG"].iloc[start:start+WINDOW].values

    rms_list.append(compute_rms(env))
    mdf_list.append(compute_mdf(raw, FS))

rms_list = np.array(rms_list)
mdf_list = np.array(mdf_list)

# Trends (slopes)
time_axis = np.arange(len(rms_list)).reshape(-1, 1)
rms_slope = LinearRegression().fit(time_axis, rms_list).coef_[0]
mdf_slope = LinearRegression().fit(time_axis, mdf_list).coef_[0]

# Baselines
mean_rms = rms_list.mean()
mean_mdf = mdf_list.mean()

print("=== Dataset Reference Built ===")
print(f"RMS mean  : {mean_rms:.2f}")
print(f"MDF mean  : {mean_mdf:.2f}")
print(f"RMS slope : {rms_slope:.6f}")
print(f"MDF slope : {mdf_slope:.6f}")



=== Dataset Reference Built ===
RMS mean  : 216.70
MDF mean  : 46.70
RMS slope : -1.037950
MDF slope : -0.012096


In [19]:
# -------------------------------------------------
# SERIAL SETUP
# -------------------------------------------------
ser = serial.Serial(PORT, BAUD, timeout=1)
time.sleep(2)
print("\nListening to live EMG... (Ctrl+C to stop)\n")

raw_buffer = deque(maxlen=WINDOW)
env_buffer = deque(maxlen=WINDOW)

fatigue_counter=0
t1=0

# LIVE LOOP



Listening to live EMG... (Ctrl+C to stop)



In [20]:
fatigue_counter=0
t1=0
try:
    while True:
        line = ser.readline().decode(errors="ignore").strip()

        if "," not in line:
            continue

        try:
            raw, env = map(float, line.split(","))
        except ValueError:
            continue

        raw_buffer.append(raw)
        env_buffer.append(env)

        if len(raw_buffer) < WINDOW:
            continue

        # Compute live features
        live_rms = compute_rms(np.array(env_buffer))
        live_mdf = compute_mdf(np.array(raw_buffer), FS)

        # Fatigue logic
        fatigued = (
            live_rms > mean_rms and
            live_mdf < mean_mdf
        )

        status = "⚠️ FATIGUED" if fatigued else "✅ NON-FATIGUED"

        print(
            f"Live RMS: {live_rms:7.2f} | "
            f"Live MDF: {live_mdf:6.1f} Hz | "
            f"RMS slope : {live_rms:.6f} |"
            f"MDF slope : {live_mdf:.6f} |"

            f"{status}"
        )
        if fatigued:
            fatigue_counter += 1
            print(f"Fatigue window detected ({fatigue_counter}/600)")
            t1+=1

        # If fatigue persists for 3 seconds
        if fatigue_counter >= 600:
            print("\n⚠️ FATIGUE DETECTED (for 3 seconds)")
            fatigue_counter=0
            break
        if t1>=2000:
            print("Fatigue Counter restarted as 10 seconds has passedd from first detection")
            fatigue_counter=0
            t1=0
except KeyboardInterrupt:
    print("\nStopped by user")
    ser.close()

Live RMS:   30.45 | Live MDF:   48.0 Hz | RMS slope : 30.453571 |MDF slope : 48.000000 |✅ NON-FATIGUED
Live RMS:   30.45 | Live MDF:   48.0 Hz | RMS slope : 30.453571 |MDF slope : 48.000000 |✅ NON-FATIGUED
Live RMS:   30.45 | Live MDF:   48.0 Hz | RMS slope : 30.453571 |MDF slope : 48.000000 |✅ NON-FATIGUED
Live RMS:   30.45 | Live MDF:   48.0 Hz | RMS slope : 30.453571 |MDF slope : 48.000000 |✅ NON-FATIGUED
Live RMS:   30.45 | Live MDF:   48.0 Hz | RMS slope : 30.453571 |MDF slope : 48.000000 |✅ NON-FATIGUED
Live RMS:   30.46 | Live MDF:   48.0 Hz | RMS slope : 30.463092 |MDF slope : 48.000000 |✅ NON-FATIGUED
Live RMS:   30.47 | Live MDF:   48.0 Hz | RMS slope : 30.472611 |MDF slope : 48.000000 |✅ NON-FATIGUED
Live RMS:   30.48 | Live MDF:   48.0 Hz | RMS slope : 30.482126 |MDF slope : 48.000000 |✅ NON-FATIGUED
Live RMS:   30.49 | Live MDF:   48.0 Hz | RMS slope : 30.491638 |MDF slope : 48.000000 |✅ NON-FATIGUED
Live RMS:   30.50 | Live MDF:   48.0 Hz | RMS slope : 30.501148 |MDF slop